# Modélisation baseline

## Informations générales

| Élément | Détail |
|---|---|
| **Propriétaire** | Tiéba Bamba |
| **Projet** | Talk to my data |
| **Domaine** | Banque de détail - Recouvrement & Risque |
| **Étape** | 2/3 - Modélisation baseline |
| **Notebook** | Construction et évaluation d'un modèle de référence |
| **Date de création** | 25 août 2026 |
| **Statut** | En cours |

## Contexte métier

La direction **« Recouvrement & Risque »** d'une banque de détail souhaite renforcer sa politique de relance et de recouvrement au prochain trimestre. L'étape précédente (EDA) a permis de comprendre la structure du dataset, d'identifier la variable cible `default_payment_next_month`, son déséquilibre (~21 % de défauts), et les variables les plus liées au risque (notamment les statuts de remboursement `pay_*`).

Cette deuxième étape vise à construire un premier modèle de classification simple, servant de **référence (baseline)** pour évaluer l'apport de modèles plus complexes lors de l'étape suivante.

## Objectifs de ce notebook

Ce notebook constitue la deuxième étape du projet. Il a pour objectifs de :

- charger les données préparées à partir des constats de l'étape d'EDA ;
- mettre en place le prétraitement nécessaire (encodage, gestion des modalités regroupées) ;
- entraîner un modèle de classification simple et interprétable ;
- évaluer ses performances avec des métriques adaptées au déséquilibre des classes (F1, Recall, PR-AUC, matrice de confusion) ;
- établir une base de comparaison chiffrée pour la modélisation avancée à venir ;
- documenter les limites de cette première approche.

## Déroulé annoncé

1. **Initialisation** : imports et chargement des données préparées (train/test).
2. **Prétraitement** : encodage des variables catégorielles, mise en forme des features.
3. **Entraînement du modèle baseline** : choix et entraînement d'un modèle simple.
4. **Évaluation** : métriques adaptées au déséquilibre des classes, matrice de confusion.
5. **Synthèse** : constats, limites du modèle baseline et pistes pour la modélisation avancée.

## Résultat attendu

À la fin de ce notebook, nous disposerons d'un modèle de référence évalué et documenté, servant de point de comparaison pour les modèles plus avancés développés à l'étape suivante.

# Préparation des données

la préparation des données a été faite dans data_prep.py en reprenant tout les regroupement que nous avons fait en EDA.

celle ci s'est faite en différentes etapes :

### 1. Configuration (`src/config.py`)

Avant de préparer les données, on centralise dans `config.py` toutes les constantes décidées lors de l'EDA (chemins, colonnes, regroupements de modalités, paramètres de split), pour que `data_prep.py`, `train.py` et `infer.py` s'y réfèrent au lieu de dupliquer ces choix.

**Chemins et colonnes cible/exclues :**

```python
from pathlib import Path

# Chemins
PROJECT_ROOT = Path(__file__).resolve().parent.parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "credit_card_default.csv"

# Colonnes
TARGET_COLUMN = "default_payment_next_month"
ID_COLUMN = "id"
# Colonne suspecte reperee en EDA (prediction deja presente dans les donnees brutes,
# risque de fuite de donnees) : a exclure des features.
LEAKAGE_COLUMN = "predicted_default_payment_next_month"
EXCLUDED_COLUMNS = [ID_COLUMN, LEAKAGE_COLUMN]
```

`id` n'est pas une feature, et `predicted_default_payment_next_month` a été identifiée en EDA comme une colonne à risque de fuite de données (prédiction déjà présente dans les données brutes, réservée aux contrôles qualité selon les consignes du projet) : les deux sont exclues.

**Colonnes catégorielles/numériques et regroupements de modalités :**

```python
CATEGORICAL_COLUMNS = ["sex", "marital_status", "education_level"]
TYPE_CAST_COLUMNS = ["sex", "marital_status"]
NUMERIC_COLUMNS = [
    "limit_balance", "age",
    "pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6",
    "bill_amt_1", "bill_amt_2", "bill_amt_3", "bill_amt_4", "bill_amt_5", "bill_amt_6",
    "pay_amt_1", "pay_amt_2", "pay_amt_3", "pay_amt_4", "pay_amt_5", "pay_amt_6",
]

# Regroupement des modalites decide en EDA (codes non documentes / rares)
EDUCATION_LEVEL_MAPPING = {4: 0, 5: 0, 6: 0}
MARITAL_STATUS_MAPPING = {"0": "3"}
```

Ces listes définissent explicitement quelles colonnes seront one-hot encodées vs passées telles quelles, et les mappings reprennent les regroupements de modalités décidés en EDA (`education_level` 4/5/6 → 0, `marital_status` 0 → 3).

**Paramètres de split :**

```python
TEST_SIZE = 0.2
RANDOM_STATE = 42
```

Les mêmes valeurs que dans le notebook d'EDA, pour garder un split reproductible et cohérent entre les étapes du projet.

### 2. Préparation des données (`src/data_prep.py`)

Le fichier `data_prep.py` contient 3 fonctions qui s'appuient sur `config.py`.

**`load_and_prepare_data()`** — charge le CSV brut, type les colonnes catégorielles et split train/test :

```python
def load_and_prepare_data():
    """Charge le CSV brut, type les colonnes categorielles et split train/test."""
    df = pd.read_csv(RAW_DATA_PATH)
    df[TYPE_CAST_COLUMNS] = df[TYPE_CAST_COLUMNS].astype("str")

    df_train, df_test = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df[TARGET_COLUMN],
    )
    return df_train, df_test
```

C'est l'équivalent exact des étapes faites manuellement dans le notebook d'EDA (lecture, typage `sex`/`marital_status`, split stratifié), mais centralisé pour être réutilisé partout.

**`clean_categoricals(X)`** — applique le regroupement des modalités décidé en EDA :

```python
def clean_categoricals(X):
    """Regroupe les modalites non documentees/rares (regle fixe decidee en EDA).

    Cast egalement sex/marital_status en str : rend cette etape auto-suffisante
    pour que la pipeline reste correcte meme appliquee a des donnees brutes
    (infer.py) qui n'auraient pas transite par load_and_prepare_data().
    """
    X = X.copy()
    X["sex"] = X["sex"].astype(str)
    X["marital_status"] = X["marital_status"].astype(str).replace(MARITAL_STATUS_MAPPING)
    X["education_level"] = X["education_level"].replace(EDUCATION_LEVEL_MAPPING)
    return X
```

Cette fonction ne dépend d'aucune statistique calculée sur les données (règle fixe, pas de risque de fuite train/test) : elle peut donc être appliquée telle quelle sur train, test, ou de nouvelles données à l'inférence.

**`build_pipeline(model)`** — assemble le nettoyage, l'encodage, le scaling et le modèle en une seule pipeline sklearn :

```python
def build_pipeline(model):
    """Construit la pipeline complete : nettoyage categoriel, encodage, modele."""
    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLUMNS),
            ("numeric", StandardScaler(), NUMERIC_COLUMNS),
        ]
    )

    return Pipeline(
        steps=[
            ("clean_categoricals", FunctionTransformer(clean_categoricals)),
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
```

Paramétrée par le modèle, cette fonction sera réutilisée telle quelle dans le notebook 03 (modélisation avancée) avec un autre estimateur. Le `ColumnTransformer` ne sélectionne que `CATEGORICAL_COLUMNS`/`NUMERIC_COLUMNS`, donc `id` et `predicted_default_payment_next_month` sont automatiquement exclus des features sans avoir besoin de les dropper explicitement.

Les colonnes numériques passent par un `StandardScaler()` (moyenne 0, écart-type 1) plutôt qu'un simple passage direct : sans ça, `LogisticRegression` ne convergeait pas correctement (`ConvergenceWarning`), les variables comme `limit_balance` (jusqu'à 800 000) écrasant numériquement des variables comme `pay_0` (entre -2 et 8) pendant l'optimisation. Le scaling n'a aucun effet négatif pour des modèles à base d'arbres qu'on pourrait tester en notebook 03.

### 3. Modele de base

Pour l'implémentation du modele de base nous lons utilser un modele linéaire simple qui est la regression logistique. ( ``LogisticRegression`` de ``scikit-learn`` )

In [8]:
# Imports des Variables et fonctions etablies dans les fichiers src/config.py et src/data_prep.py
import sys
sys.path.append("..")

from src.config import TARGET_COLUMN
from src.data_prep import load_and_prepare_data, build_pipeline

In [9]:
# modele logistique
from sklearn.linear_model import LogisticRegression

model_baseline = LogisticRegression(random_state=42 , max_iter=1000, class_weight="balanced")


In [10]:
import pandas as pd

df_train, df_test = load_and_prepare_data()

X_train, y_train = df_train.drop(columns=[TARGET_COLUMN]), df_train[TARGET_COLUMN]
X_test, y_test = df_test.drop(columns=[TARGET_COLUMN]), df_test[TARGET_COLUMN]

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}")

X_train : (2372, 25) | X_test : (593, 25)


In [11]:
from sklearn import set_config
set_config(display="diagram")

pipeline_baseline = build_pipeline(model_baseline)
pipeline_baseline.fit(X_train, y_train)
pipeline_baseline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('clean_categoricals', ...), ('preprocessor', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](25,)","['id','limit_balance','sex',...,'pay_amt_5','pay_amt_6', 'predicted_default_payment_next_month']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,25
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function cle...0029796AEC5C0>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False


### 4. Évaluation

L'accuracy seule n'est pas adaptée ici : vu le déséquilibre des classes (~21 % de défauts), un modèle qui prédit toujours "pas de défaut" aurait déjà ~79 % d'accuracy sans aucune valeur métier. On utilise donc des métriques plus pertinentes pour ce contexte de risque :

- **Recall** : proportion des vrais défauts effectivement détectés — c'est le plus important ici, puisque manquer un client à risque coûte cher.
- **Precision** : proportion des alertes de défaut qui sont correctes.
- **F1-score** : compromis entre les deux.
- **PR-AUC** : performance globale du score de probabilité, plus informative que la ROC-AUC pour des classes déséquilibrées.
- **Matrice de confusion** : pour visualiser concrètement les erreurs (faux positifs vs faux négatifs).

In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
)

y_pred = pipeline_baseline.predict(X_test)
y_proba = pipeline_baseline.predict_proba(X_test)[:, 1]

print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision : {precision_score(y_test, y_pred):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred):.3f}")
print(f"F1-score  : {f1_score(y_test, y_pred):.3f}")
print(f"PR-AUC    : {average_precision_score(y_test, y_proba):.3f}  (vs {y_test.mean():.3f} pour un modele aleatoire)")

Accuracy  : 0.642
Precision : 0.328
Recall    : 0.638
F1-score  : 0.433
PR-AUC    : 0.491  (vs 0.214 pour un modele aleatoire)


In [13]:
y_pred_train = pipeline_baseline.predict(X_train)
y_proba_train = pipeline_baseline.predict_proba(X_train)[:, 1]

print("--- Performance sur le train ---")
print(f"Accuracy  : {accuracy_score(y_train, y_pred_train):.3f}")
print(f"Precision : {precision_score(y_train, y_pred_train):.3f}")
print(f"Recall    : {recall_score(y_train, y_pred_train):.3f}")
print(f"F1-score  : {f1_score(y_train, y_pred_train):.3f}")
print(f"PR-AUC    : {average_precision_score(y_train, y_proba_train):.3f}")

--- Performance sur le train ---
Accuracy  : 0.708
Precision : 0.397
Recall    : 0.699
F1-score  : 0.506
PR-AUC    : 0.566


**Train vs test** :

| Métrique | Train | Test |
|---|---|---|
| Accuracy | 0,708 | 0,642 |
| Precision | 0,397 | 0,328 |
| Recall | 0,699 | 0,638 |
| F1-score | 0,506 | 0,433 |
| PR-AUC | 0,566 | 0,491 |

L'écart entre train et test reste modéré (quelques points sur chaque métrique), ce qui est cohérent avec un modèle linéaire simple : peu de capacité à sur-apprendre le bruit du train. Pas de signe d'overfitting marqué ici — la marge de progression viendra plus probablement d'un modèle avec plus de capacité (notebook 03) que d'une régularisation du baseline.

In [16]:
import plotly.graph_objects as go
from IPython.display import HTML, display

cm = confusion_matrix(y_test, y_pred)

fig = go.Figure(
    data=go.Heatmap(
        z=cm,
        x=["Pas de défaut (prédit)", "Défaut (prédit)"],
        y=["Pas de défaut (réel)", "Défaut (réel)"],
        colorscale="Blues",
        text=cm,
        texttemplate="%{text}",
        textfont={"size": 16},
        colorbar={"title": "Effectif"},
    )
)

fig.update_layout(
    title="Matrice de confusion - Modèle baseline",
    template="plotly_white",
    height=500,
    width=550,
)

display(HTML(fig.to_html(include_plotlyjs="cdn")))

##### **Interprétation des résultats**

- **Accuracy (0,642)** : nettement plus basse que si on n'avait pas géré le déséquilibre (~0,82 sans `class_weight="balanced"`), mais c'est attendu et volontaire — `class_weight="balanced"` pousse le modèle à mieux détecter la classe minoritaire au prix de plus d'erreurs sur la majoritaire.
- **Recall (0,638)** : le modèle détecte environ **64 %** des vrais défauts, contre ~0 % pour un modèle naïf qui prédirait toujours "pas de défaut". C'est la métrique la plus importante dans un contexte de recouvrement : mieux vaut relancer un client qui n'était pas à risque que manquer un vrai défaut.
- **Precision (0,328)** : parmi les clients signalés à risque, environ 1 sur 3 fait effectivement défaut. Beaucoup de faux positifs (166 sur la matrice de confusion), ce qui a un coût opérationnel (relances inutiles) à mettre en balance avec le coût d'un défaut manqué.
- **PR-AUC (0,491)** vs **0,214** pour un modèle aléatoire (le taux de défaut) : le modèle capture un signal réel, plus de deux fois meilleur qu'un score aléatoire, mais reste loin d'un score parfait (1,0).
- **Matrice de confusion** : 81 vrais défauts détectés, 46 manqués, 166 fausses alertes, 300 vrais négatifs.

**Limite principale de ce baseline** : le compromis precision/recall est très marqué (beaucoup de faux positifs pour capter plus de vrais défauts), ce qui reflète la simplicité du modèle linéaire et l'absence de traitement plus fin des variables (ex. multicolinéarité des `bill_amt_*`, encodage ordinal de `education_level`, seuil de décision par défaut à 0,5 non optimisé). Ce score sert de **point de comparaison chiffré** pour le notebook 03 (modélisation avancée).

### 5. Ajustement du seuil de décision

Par défaut, `predict()` classe en "défaut" tout client dont la probabilité prédite dépasse 0,5. Ce seuil est arbitraire et ne tient pas compte du coût métier : dans un contexte de recouvrement, **manquer un vrai défaut coûte plus cher qu'une fausse alerte** (relance inutile). Il est donc légitime d'abaisser le seuil pour privilégier le Recall, au prix d'une Precision plus faible.

Ce réglage se fait directement sur les probabilités déjà prédites (`y_proba`), sans réentraîner le modèle — complémentaire à `class_weight="balanced"` (qui agit pendant l'entraînement), pas redondant avec lui.

In [15]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

fig = go.Figure()
fig.add_trace(go.Scatter(x=thresholds, y=precisions[:-1], name="Precision", mode="lines"))
fig.add_trace(go.Scatter(x=thresholds, y=recalls[:-1], name="Recall", mode="lines"))

fig.update_layout(
    title="Precision et Recall en fonction du seuil de décision",
    xaxis_title="Seuil",
    yaxis_title="Score",
    template="plotly_white",
    height=500,
    width=800,
)

display(HTML(fig.to_html(include_plotlyjs="cdn")))

**Choix du seuil** : on fixe un objectif métier explicite plutôt qu'une valeur arbitraire — ici, détecter au moins **80 % des vrais défauts** (Recall ≥ 0,8). Le seuil le plus élevé qui satisfait cette contrainte est choisi, pour ne pas sacrifier plus de Precision que nécessaire.

In [17]:
import numpy as np

target_recall = 0.8
eligible = np.where(recalls[:-1] >= target_recall)[0]
best_idx = eligible[np.argmax(thresholds[eligible])]
chosen_threshold = thresholds[best_idx]

print(f"Seuil choisi : {chosen_threshold:.3f}")
print(f"Precision a ce seuil : {precisions[best_idx]:.3f}")
print(f"Recall a ce seuil    : {recalls[best_idx]:.3f}")

y_pred_adjusted = (y_proba >= chosen_threshold).astype(int)

print(f"\nAccuracy  : {accuracy_score(y_test, y_pred_adjusted):.3f}")
print(f"Precision : {precision_score(y_test, y_pred_adjusted):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred_adjusted):.3f}")
print(f"F1-score  : {f1_score(y_test, y_pred_adjusted):.3f}")

Seuil choisi : 0.323
Precision a ce seuil : 0.247
Recall a ce seuil    : 0.803

Accuracy  : 0.433
Precision : 0.247
Recall    : 0.803
F1-score  : 0.378


In [18]:
cm_adjusted = confusion_matrix(y_test, y_pred_adjusted)

fig = go.Figure(
    data=go.Heatmap(
        z=cm_adjusted,
        x=["Pas de défaut (prédit)", "Défaut (prédit)"],
        y=["Pas de défaut (réel)", "Défaut (réel)"],
        colorscale="Blues",
        text=cm_adjusted,
        texttemplate="%{text}",
        textfont={"size": 16},
        colorbar={"title": "Effectif"},
    )
)

fig.update_layout(
    title=f"Matrice de confusion - Seuil ajusté ({chosen_threshold:.2f})",
    template="plotly_white",
    height=500,
    width=550,
)

display(HTML(fig.to_html(include_plotlyjs="cdn")))

##### **Interprétation du seuil ajusté**

Avec un seuil de **0,323** (au lieu de 0,5) :

| Métrique | Seuil 0,5 | Seuil 0,323 |
|---|---|---|
| Accuracy | 0,642 | 0,433 |
| Precision | 0,328 | 0,247 |
| Recall | 0,638 | **0,803** |
| F1-score | 0,433 | 0,378 |

Le Recall passe de 64 % à **80 %** : le modèle détecte maintenant 102 vrais défauts sur 127 (25 manqués au lieu de 46). Le prix à payer est net — la Precision tombe à 0,247 (311 fausses alertes contre 166 avant) et l'accuracy chute à 0,433, en dessous du seuil de 0,5 "au hasard". C'est le résultat attendu et cohérent avec l'objectif fixé (Recall ≥ 0,8), pas un signe que le modèle s'est dégradé.

**Point d'attention pour la suite** : ce compromis n'est acceptable que si le coût réel d'un défaut manqué justifie ce volume de fausses alertes (311 relances inutiles sur 593 clients test). C'est une décision métier, pas seulement technique — idéalement à valider avec la direction Recouvrement & Risque avec de vrais coûts ($ par relance vs $ par défaut manqué) plutôt qu'un objectif de Recall choisi arbitrairement à 80 %. Le F1-score, qui baisse ici (0,378 vs 0,433), rappelle que ce choix de seuil n'optimise pas un compromis équilibré mais privilégie délibérément le Recall.